In [1]:
import json

docs = []
file_path = r"C:\Users\VINIL\Documents\RedrobHackathon\data\data2\candidates.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        docs.append(data)

In [2]:
import json


def candidate_to_document(candidate):
    profile = candidate.get("profile", {})
    career_history = candidate.get("career_history", [])
    education = candidate.get("education", [])
    skills = candidate.get("skills", [])
    languages = candidate.get("languages", [])
    signals = candidate.get("redrob_signals", {})

    # -------------------------
    # DESCRIPTION
    # -------------------------

    parts = []

    parts.append(f"Candidate ID: {candidate.get('candidate_id', '')}")

    parts.append(
        f"""
Name: {profile.get('anonymized_name', '')}
Headline: {profile.get('headline', '')}

Summary:
{profile.get('summary', '')}
""".strip()
    )

    parts.append(
        f"""
Current Role:
{profile.get('current_title', '')} at {profile.get('current_company', '')}
Industry: {profile.get('current_industry', '')}
Location: {profile.get('location', '')}
""".strip()
    )

    # Career History
    career_text = ["Career History:"]
    for job in career_history:
        career_text.append(
            f"""
{job.get('title', '')} | {job.get('company', '')}
{job.get('start_date', '')} - {job.get('end_date', 'Present')}
{job.get('description', '')}
""".strip()
        )

    parts.append("\n\n".join(career_text))

    # Education
    education_text = ["Education:"]
    for edu in education:
        education_text.append(
            f"""
{edu.get('degree', '')} in {edu.get('field_of_study', '')}
{edu.get('institution', '')}
{edu.get('start_year', '')}-{edu.get('end_year', '')}
Grade: {edu.get('grade', '')}
""".strip()
        )

    parts.append("\n\n".join(education_text))

    # Skills
    skill_text = ["Skills:"]
    for skill in skills:
        skill_text.append(
            f"{skill.get('name', '')} ({skill.get('proficiency', '')})"
        )

    parts.append("\n".join(skill_text))

    # Languages
    lang_text = ["Languages:"]
    for lang in languages:
        lang_text.append(
            f"{lang.get('language', '')} ({lang.get('proficiency', '')})"
        )

    parts.append("\n".join(lang_text))

    description = "\n\n".join(parts)

    # -------------------------
    # METADATA
    # -------------------------

    highest_degree = None
    if education:
        degree_priority = {
            "Ph.D": 5,
            "Doctorate": 5,
            "Master": 4,
            "M.Tech": 4,
            "MBA": 4,
            "B.Tech": 3,
            "B.E": 3,
            "B.Sc": 3,
            "Bachelor": 3,
            "Diploma": 2
        }

        highest_degree = max(
            education,
            key=lambda x: degree_priority.get(x.get("degree", ""), 0)
        ).get("degree")

    salary = signals.get("expected_salary_range_inr_lpa", {})

    metadata = {
        # IDs
        "candidate_id": candidate.get("candidate_id"),

        # Experience
        "years_experience": profile.get("years_of_experience"),

        # Location
        "country": profile.get("country"),
        "location": profile.get("location"),

        # Current Role
        "current_title": profile.get("current_title"),
        "current_company": profile.get("current_company"),
        "industry": profile.get("current_industry"),

        # Education
        "highest_degree": highest_degree,

        # Skills
        "skills": [s.get("name") for s in skills],

        # Signals
        "open_to_work": signals.get("open_to_work_flag"),
        "notice_period_days": signals.get("notice_period_days"),
        "willing_to_relocate": signals.get("willing_to_relocate"),
        "preferred_work_mode": signals.get("preferred_work_mode"),

        # Salary
        "salary_min_lpa": salary.get("min"),
        "salary_max_lpa": salary.get("max"),

        # Scores
        "profile_completeness": signals.get("profile_completeness_score"),
        "response_rate": signals.get("recruiter_response_rate"),
        "interview_completion_rate": signals.get("interview_completion_rate"),
        "offer_acceptance_rate": signals.get("offer_acceptance_rate"),

        # Counts
        "connection_count": signals.get("connection_count"),
        "applications_30d": signals.get("applications_submitted_30d"),
        "profile_views_30d": signals.get("profile_views_received_30d"),

        # Verification
        "verified_email": signals.get("verified_email"),
        "verified_phone": signals.get("verified_phone"),
    }

    return {
        "description": description,
        "metadata": metadata
    }

In [3]:
documents = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        candidate = json.loads(line)
        documents.append(candidate_to_document(candidate))

In [4]:
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from qdrant_client.models import PointStruct

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = QdrantClient(path="./qdrant_d_b")

points = []
for d in documents:
    text = d["description"]
    embedding = model.encode(text)
    point = PointStruct(
        id = d["metadata"]["candidate_id"],
        vector = embedding.tolist(),
        payload = d["metadata"]
    )
    points.append(point)  
    print(d["metadata"]["candidate_id"])

client.upsert(
    collection_name="candidates",
    points=points)

c:\Users\VINIL\Documents\RedrobHackathon\Environment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10933.11it/s]


CAND_0000001
CAND_0000002
CAND_0000003
CAND_0000004
CAND_0000005
CAND_0000006
CAND_0000007
CAND_0000008
CAND_0000009
CAND_0000010
CAND_0000011
CAND_0000012
CAND_0000013
CAND_0000014
CAND_0000015
CAND_0000016
CAND_0000017
CAND_0000018
CAND_0000019
CAND_0000020
CAND_0000021
CAND_0000022
CAND_0000023
CAND_0000024
CAND_0000025
CAND_0000026
CAND_0000027
CAND_0000028
CAND_0000029
CAND_0000030
CAND_0000031
CAND_0000032
CAND_0000033
CAND_0000034
CAND_0000035
CAND_0000036
CAND_0000037
CAND_0000038
CAND_0000039
CAND_0000040
CAND_0000041
CAND_0000042
CAND_0000043
CAND_0000044
CAND_0000045
CAND_0000046
CAND_0000047
CAND_0000048
CAND_0000049
CAND_0000050
CAND_0000051
CAND_0000052
CAND_0000053
CAND_0000054
CAND_0000055
CAND_0000056
CAND_0000057
CAND_0000058
CAND_0000059
CAND_0000060
CAND_0000061
CAND_0000062
CAND_0000063
CAND_0000064
CAND_0000065
CAND_0000066
CAND_0000067
CAND_0000068
CAND_0000069
CAND_0000070
CAND_0000071
CAND_0000072
CAND_0000073
CAND_0000074
CAND_0000075
CAND_0000076
CAND_0000077

ValueError: Collection candidates not found

In [5]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="candidates",
    vectors_config=VectorParams(
        size=384,  # bge-small dimension
        distance=Distance.COSINE
    )
)

True

In [7]:
for point in points:
    point.id = int(
        point.payload["candidate_id"].split("_")[1]
    )
client.upsert(
    collection_name="candidates",
    points=points
)

C:\Users\VINIL\AppData\Local\Temp\ipykernel_18944\1503205911.py:5: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 100000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
import os
from qdrant_client import QdrantClient
from dotenv import load_dotenv
load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")    
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
# Local DB
local_client = QdrantClient(path="./qdrant_d_b")

# Cloud DB
cloud_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

C:\Users\VINIL\AppData\Local\Temp\ipykernel_4476\2390539944.py:8: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <candidates> contains 100000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  local_client = QdrantClient(path="./qdrant_d_b")


In [14]:
from qdrant_client.models import PointStruct

offset = None

while True:
    records, offset = local_client.scroll(
        collection_name="candidates",
        limit=100,
        offset=offset,
        with_vectors=True,
        with_payload=True,
    )

    if not records:
        break

    points = [
        PointStruct(
            id=r.id,
            vector=r.vector,
            payload=r.payload
        )
        for r in records
    ]

    cloud_client.upsert(
        collection_name="Candidates_Redrob",
        points=points
    )

    if offset is None:
        break